# 05 - Feature Engineering : Silver → Gold

**Architecture Medallion – Couche Gold**

Ce notebook construit la table ML-ready `gold.features_communes` en :
1. Agrégeant les résultats électoraux par **commune × année × scrutin** et définissant la **variable cible** (bloc dominant : Gauche / Centre / Droite)
2. Joignant les features socio-économiques (CSP, diplômes, naissances, décès, revenus)
3. Créant des features temporelles (tendances, lags)

### Variable cible : Bloc dominant

| Bloc | Nuances incluses |
|------|------------------|
| **Gauche** | EXG, LEXG, LXG, COM, LCOM, SOC, LSOC, DVG, LDVG, FG, LFG, NUP, LFI, FI, ECO, LECO, VEC, LVEC, MELE, HOLL, ROYA, BUFF, VOYN, BESA, ARTH, POUT, GLUC, HUE, JOSP, LO, LCR, PG, LPG, BC-EXG, BC-DVG, BC-FG, BC-SOC, BC-FI, BC-COM, BC-VEC, BC-ECO, BC-PG, BC-UCG, DXG, PREP, GAU |
| **Centre** | ENS, REM, LREM, BAYR, UDF, LUDF, MODM, LMDM, MDM, MDC, UDI, LUDI, LUD, UG, LUG, LUC, DVC, LDVC, LCMD, LUGE, BC-REM, BC-MDM, BC-UDI, BC-UC, BC-UCD, NCE, M, M-NC |
| **Droite** | UMP, LUMP, LR, LLR, SARK, DVD, LDVD, CHIR, FILLON, LEPE (RPR → LR), RPR, RPF, DLF, LDLF, BC-UMP, BC-LR, BC-DVD, BC-DLF, MPF, NCE, BC-UC, MADE |
| **Extrême droite** | FN, LFN, RN, LRN, MEGR, MNR, LEPA, BC-FN, BC-RN, EXD, LEXD, LXD, BC-EXD, ZEMMOUR, REC, LREC |
| **Divers** | DIV, LDIV, LDV, AUT, LAUT, LDIV, BC-DIV, REG |

**Agrégation** : Pour les présidentielles (t1) on calcule le % de voix par bloc. Le bloc dominant = argmax(% voix).
On utilise **présidentielles t1** comme scrutin de référence (disponible 2002→2022, tous les 5 ans).

## 0. Configuration & imports

In [1]:
import os
import pandas as pd
import numpy as np
from sqlalchemy import create_engine, text

# Configuration inline
PG_HOST     = os.environ.get("POSTGRES_HOST",     "postgres")
PG_PORT     = os.environ.get("POSTGRES_PORT",     "5432")
PG_DB       = os.environ.get("POSTGRES_DB",       "mspr813")
PG_USER     = os.environ.get("POSTGRES_USER",     "mspr_user")
PG_PASSWORD = os.environ.get("POSTGRES_PASSWORD", "mspr_password")

DB_URL = f"postgresql+psycopg2://{PG_USER}:{PG_PASSWORD}@{PG_HOST}:{PG_PORT}/{PG_DB}"
engine = create_engine(DB_URL, pool_pre_ping=True)

with engine.connect() as conn:
    print("PostgreSQL OK :", conn.execute(text("SELECT current_database()")).scalar())

PostgreSQL OK : mspr813


## 1. Schéma Gold

In [2]:
with engine.begin() as conn:
    conn.execute(text("""
        CREATE SCHEMA IF NOT EXISTS gold;

        DROP TABLE IF EXISTS gold.features_communes CASCADE;

        CREATE TABLE gold.features_communes (
            -- Clé
            code_commune        CHAR(5)      NOT NULL,
            annee               SMALLINT     NOT NULL,
            libelle             VARCHAR(100),
            code_dep            CHAR(3),

            -- Variable cible (scrutin présidentiel t1)
            pct_gauche          NUMERIC(6,3),
            pct_centre          NUMERIC(6,3),
            pct_droite          NUMERIC(6,3),
            pct_divers          NUMERIC(6,3),
            bloc_dominant       VARCHAR(20),   -- Gauche/Centre/Droite/Divers

            -- Features démographiques
            nb_naissances       INTEGER,
            nb_deces            INTEGER,
            solde_naturel       INTEGER,       -- naissances - décès

            -- Features revenus
            mediane_revenu_disp NUMERIC(10,2),
            gini                NUMERIC(6,4),
            taux_pauvrete       NUMERIC(6,3),

            -- Features CSP (2022 seulement, répliqué sur toutes les années)
            cadres_pct          NUMERIC(6,3),  -- cadres_emploi / total_emploi
            ouvriers_pct        NUMERIC(6,3),
            employes_pct        NUMERIC(6,3),
            artisans_pct        NUMERIC(6,3),

            -- Features diplômes (2022 seulement, répliqué)
            pct_bac_plus        NUMERIC(6,3),  -- (sup2+sup34+sup5) / nscol15p
            pct_sans_diplome    NUMERIC(6,3),  -- sans_diplome / nscol15p

            -- Métadonnées
            created_at          TIMESTAMP DEFAULT NOW(),
            CONSTRAINT pk_gold_features PRIMARY KEY (code_commune, annee)
        );
    """))
print("Schéma gold créé")

Schéma gold créé


## 2. Mapping nuance → bloc politique

In [3]:
# Mapping nuance_liste -> bloc politique
NUANCE_TO_BLOC = {
    # ---- GAUCHE ----
    'EXG': 'Gauche', 'LEXG': 'Gauche', 'LXG': 'Gauche', 'DXG': 'Gauche',
    'COM': 'Gauche', 'LCOM': 'Gauche',
    'SOC': 'Gauche', 'LSOC': 'Gauche',
    'DVG': 'Gauche', 'LDVG': 'Gauche',
    'FG':  'Gauche', 'LFG':  'Gauche',
    'NUP': 'Gauche',
    'LFI': 'Gauche', 'FI': 'Gauche',
    'ECO': 'Gauche', 'LECO': 'Gauche',
    'VEC': 'Gauche', 'LVEC': 'Gauche',
    'LCR': 'Gauche', 'LO': 'Gauche',
    'PG':  'Gauche', 'LPG': 'Gauche',
    'BC-EXG': 'Gauche', 'BC-DVG': 'Gauche', 'BC-FG': 'Gauche',
    'BC-SOC': 'Gauche', 'BC-FI': 'Gauche', 'BC-COM': 'Gauche',
    'BC-VEC': 'Gauche', 'BC-ECO': 'Gauche', 'BC-PG': 'Gauche',
    'BC-UCG': 'Gauche',
    'GAU': 'Gauche', 'PREP': 'Gauche',
    # Présidentiels gauche
    'MELE': 'Gauche', 'HOLL': 'Gauche', 'ROYA': 'Gauche',
    'BUFF': 'Gauche', 'VOYN': 'Gauche', 'BESA': 'Gauche',
    'ARTH': 'Gauche', 'POUT': 'Gauche', 'GLUC': 'Gauche',
    'HUE': 'Gauche', 'JOSP': 'Gauche', 'BOVE': 'Gauche',
    'TAUB': 'Gauche', 'JOLY': 'Gauche',
    # Verts
    'MAME': 'Gauche',

    # ---- CENTRE ----
    'ENS': 'Centre', 'REM': 'Centre', 'LREM': 'Centre',
    'BC-REM': 'Centre', 'BC-MDM': 'Centre',
    'BAYR': 'Centre', 'LCOP': 'Centre',
    'UDF': 'Centre', 'LUDF': 'Centre',
    'MODM': 'Centre', 'LMDM': 'Centre', 'MDM': 'Centre', 'MDC': 'Centre',
    'UDI': 'Centre', 'LUDI': 'Centre', 'BC-UDI': 'Centre',
    'LUD': 'Centre',
    'UG':  'Centre', 'LUG': 'Centre', 'LUC': 'Centre', 'LUGE': 'Centre',
    'DVC': 'Centre', 'LDVC': 'Centre', 'LCMD': 'Centre',
    'BC-UC': 'Centre', 'BC-UCD': 'Centre',
    'NCE': 'Centre', 'M': 'Centre', 'M-NC': 'Centre',
    'BC-DVC': 'Centre',
    # Présidentiels centre
    'CHEV': 'Centre', 'LEPAGE': 'Centre', 'LASA': 'Centre',

    # ---- DROITE ----
    'UMP': 'Droite', 'LUMP': 'Droite', 'BC-UMP': 'Droite',
    'LR': 'Droite', 'LLR': 'Droite', 'BC-LR': 'Droite',
    'RPR': 'Droite', 'RPF': 'Droite',
    'DVD': 'Droite', 'LDVD': 'Droite', 'BC-DVD': 'Droite',
    'DLF': 'Droite', 'LDLF': 'Droite', 'BC-DLF': 'Droite',
    'MPF': 'Droite',
    'UDI': 'Droite',  # parfois classé droite selon contexte
    'UDFD': 'Droite',
    # Présidentiels droite
    'SARK': 'Droite', 'CHIR': 'Droite', 'MADE': 'Droite',
    'PECA': 'Droite', 'FILL': 'Droite',
    'VILL': 'Droite', 'NIHO': 'Droite', 'SCHI': 'Droite',
    'BOUT': 'Droite', 'DUPO': 'Droite', 'CHEM': 'Droite',
    'SAIN': 'Droite', 'LAGU': 'Droite',
    'CPNT': 'Droite',

    # ---- EXTREME DROITE ----
    'FN': 'Droite', 'LFN': 'Droite',
    'RN': 'Droite', 'LRN': 'Droite', 'BC-RN': 'Droite',
    'MNR': 'Droite', 'MEGR': 'Droite',
    'LEPA': 'Droite',
    'EXD': 'Droite', 'LEXD': 'Droite', 'LXD': 'Droite',
    'BC-EXD': 'Droite', 'BC-FN': 'Droite',
    'REC': 'Droite', 'LREC': 'Droite',  # Zemmour 2022
    'DTE': 'Droite', 'FRN': 'Droite', 'MNA': 'Droite',
    'UXD': 'Droite', 'BC-EXD': 'Droite',

    # ---- DIVERS ----
    'DIV': 'Divers', 'LDIV': 'Divers', 'LDV': 'Divers',
    'AUT': 'Divers', 'LAUT': 'Divers',
    'BC-DIV': 'Divers', 'REG': 'Divers',
    'LDVC': 'Divers',
    'LDD': 'Divers', 'LXD': 'Divers', 'DSV': 'Divers',
    'BC-RDG': 'Divers', 'RDG': 'Divers', 'PRG': 'Divers', 'PRV': 'Divers',
    'ALLI': 'Divers', 'LAGE': 'Divers',
    'LDR': 'Divers', 'LFG': 'Divers',
    'BC-UC': 'Divers',
}

# Mapping présidentiels par nom (quand nuance absente)
CANDIDAT_PRES_TO_BLOC = {
    # Gauche
    'ARTHAUD': 'Gauche', 'POUTOU': 'Gauche', 'MELENCHON': 'Gauche',
    'MÉLENCHON': 'Gauche', 'JADOT': 'Gauche', 'HIDALGO': 'Gauche',
    'ROUSSEL': 'Gauche', 'HAMON': 'Gauche', 'HOLLANDE': 'Gauche',
    'ROYAL': 'Gauche', 'JOSPIN': 'Gauche', 'BUFFET': 'Gauche',
    'VOYNET': 'Gauche', 'BESANCENOT': 'Gauche', 'GLUCKSTEIN': 'Gauche',
    'HUE': 'Gauche', 'MAMERE': 'Gauche', 'BOVÉ': 'Gauche',
    'TAUBIRA': 'Gauche', 'JOLY': 'Gauche',
    # Centre
    'MACRON': 'Centre', 'BAYROU': 'Centre', 'CHEVENEMENT': 'Centre',
    'LEPAGE': 'Centre', 'LASSALLE': 'Centre',
    # Droite
    'SARKOZY': 'Droite', 'CHIRAC': 'Droite', 'FILLON': 'Droite',
    'PÉCRESSE': 'Droite', 'MADELIN': 'Droite', 'de VILLIERS': 'Droite',
    'NIHOUS': 'Droite', 'SCHIVARDI': 'Droite', 'BOUTIN': 'Droite',
    'DUPONT-AIGNAN': 'Droite', 'SAINT-JOSSE': 'Droite', 'LAGUILLER': 'Gauche',
    'ASSELINEAU': 'Droite', 'CHEMINADE': 'Divers',
    # Extrême droite
    'LE PEN': 'Droite', 'ZEMMOUR': 'Droite',
    'MEGRET': 'Droite',
}

print(f"Mapping nuances : {len(NUANCE_TO_BLOC)} entrées")
print(f"Mapping candidats pres : {len(CANDIDAT_PRES_TO_BLOC)} entrées")

Mapping nuances : 148 entrées
Mapping candidats pres : 42 entrées


## 3. Agrégation électorale par commune × année

On utilise les **présidentielles 1er tour** comme scrutin de référence (2002, 2007, 2012, 2017, 2022).
Pour les années sans présidentielle, on interpolera/propagera dans le notebook ML.

In [4]:
# Charger toutes les élections présidentielles t1
df_elec = pd.read_sql("""
    SELECT code_commune, id_election, annee, nuance_liste, nom_candidat, nb_voix
    FROM silver.elections
    WHERE type_election = 'pres' AND tour = 1
      AND nb_voix IS NOT NULL
""", engine)

print(f"Lignes présidentielles t1 : {len(df_elec):,}")
print(f"Années : {sorted(df_elec['annee'].unique())}")
print(f"Communes : {df_elec['code_commune'].nunique()}")
df_elec.head(5)

Lignes présidentielles t1 : 205,532
Années : [2002, 2007, 2012, 2017, 2022]
Communes : 124


,code_commune,id_election,annee,nuance_liste,nom_candidat,nb_voix
0,75056,2022_pres_t1,2022,None,ARTHAUD,89
1,75056,2022_pres_t1,2022,None,ARTHAUD,1
2,75056,2022_pres_t1,2022,None,ARTHAUD,0
3,75056,2022_pres_t1,2022,None,ARTHAUD,1
4,75056,2022_pres_t1,2022,None,ARTHAUD,2


In [5]:
# Attribuer un bloc à chaque ligne
def get_bloc(row):
    # Priorité 1 : nuance_liste
    if pd.notna(row['nuance_liste']) and row['nuance_liste'] != '':
        b = NUANCE_TO_BLOC.get(row['nuance_liste'])
        if b: return b
    # Priorité 2 : nom_candidat (présidentielles)
    if pd.notna(row['nom_candidat']) and row['nom_candidat'] != '':
        b = CANDIDAT_PRES_TO_BLOC.get(row['nom_candidat'].upper().strip())
        if b: return b
        # Essai sans accents
        nom_clean = row['nom_candidat'].upper().strip()
        for k, v in CANDIDAT_PRES_TO_BLOC.items():
            if k.upper() in nom_clean or nom_clean in k.upper():
                return v
    return 'Divers'

df_elec['bloc'] = df_elec.apply(get_bloc, axis=1)

print("Distribution blocs:")
print(df_elec.groupby('bloc')['nb_voix'].sum().sort_values(ascending=False).to_string())

Distribution blocs:
bloc
Gauche    5638391
Droite    5262237
Centre    3094741
Divers       4971


In [6]:
# Agréger par commune × année × bloc
df_bloc = (
    df_elec
    .groupby(['code_commune', 'annee', 'bloc'], as_index=False)['nb_voix']
    .sum()
)

# Total voix par commune × année
df_total = (
    df_elec
    .groupby(['code_commune', 'annee'], as_index=False)['nb_voix']
    .sum()
    .rename(columns={'nb_voix': 'total_voix'})
)

df_bloc = df_bloc.merge(df_total, on=['code_commune', 'annee'])
df_bloc['pct'] = df_bloc['nb_voix'] / df_bloc['total_voix'] * 100

# Pivoter : une ligne par commune × année, une colonne par bloc
df_pivot = df_bloc.pivot_table(
    index=['code_commune', 'annee'],
    columns='bloc',
    values='pct',
    fill_value=0
).reset_index()
df_pivot.columns.name = None

# S'assurer que tous les blocs existent
for bloc in ['Gauche', 'Centre', 'Droite', 'Divers']:
    if bloc not in df_pivot.columns:
        df_pivot[bloc] = 0.0

df_pivot = df_pivot.rename(columns={
    'Gauche':        'pct_gauche',
    'Centre':        'pct_centre',
    'Droite':        'pct_droite',
    'Divers':        'pct_divers',
})

# Bloc dominant (parmi les 4 blocs principaux)
blocs_cols = ['pct_gauche', 'pct_centre', 'pct_droite']
bloc_labels = ['Gauche', 'Centre', 'Droite']
df_pivot['bloc_dominant'] = df_pivot[blocs_cols].idxmax(axis=1).map(
    dict(zip(blocs_cols, bloc_labels))
)

print(f"Shape pivot : {df_pivot.shape}")
print(f"Années : {sorted(df_pivot['annee'].unique())}")
print(f"\nDistribution blocs dominants:")
print(df_pivot.groupby(['annee','bloc_dominant']).size().to_string())
df_pivot.head(5)

Shape pivot : (620, 7)
Années : [2002, 2007, 2012, 2017, 2022]

Distribution blocs dominants:
annee  bloc_dominant
2002   Droite           103
       Gauche            21
2007   Droite            68
       Gauche            56
2012   Droite            45
       Gauche            79
2017   Centre             7
       Droite            63
       Gauche            54
2022   Centre            34
       Droite            10
       Gauche            80


,code_commune,annee,pct_centre,pct_divers,pct_droite,pct_gauche,bloc_dominant
0,75056,2002,14.534312,0.000000,48.408999,37.056689,Droite
1,75056,2007,20.731619,0.000000,41.653697,37.614684,Droite
2,75056,2012,9.336668,0.000000,39.619778,51.043554,Gauche
3,75056,2017,35.343720,0.136732,33.885463,30.634085,Centre
4,75056,2022,36.486860,0.000000,21.199892,42.313248,Gauche


## 4. Jointure features socio-économiques

In [7]:
# --- Référentiel communes ---
df_ref = pd.read_sql("""
    SELECT code_insee AS code_commune, libelle, code_dep
    FROM silver.referentiel_communes
""", engine)

# --- Naissances ---
df_nais = pd.read_sql("""
    SELECT code_commune, annee, nb_naissances
    FROM silver.naissances
""", engine)

# --- Décès ---
df_deces = pd.read_sql("""
    SELECT code_commune, annee, nb_deces
    FROM silver.deces
""", engine)

# --- Revenus ---
df_rev = pd.read_sql("""
    SELECT code_commune, annee, mediane_revenu_disp, gini, taux_pauvrete
    FROM silver.revenus
""", engine)

# --- CSP (2022 uniquement) ---
df_csp = pd.read_sql("""
    SELECT code_commune,
           cadres_emploi, ouvriers_emploi, employes_emploi, artisans_emploi,
           prof_interm_emploi,
           COALESCE(cadres_emploi,0) + COALESCE(ouvriers_emploi,0) +
           COALESCE(employes_emploi,0) + COALESCE(artisans_emploi,0) +
           COALESCE(prof_interm_emploi,0) + COALESCE(agriculteurs_emploi,0) AS total_emploi
    FROM silver.csp
    WHERE annee = 2022
""", engine)

# Calculer les % CSP
for col, new_col in [('cadres_emploi','cadres_pct'),('ouvriers_emploi','ouvriers_pct'),
                      ('employes_emploi','employes_pct'),('artisans_emploi','artisans_pct')]:
    df_csp[new_col] = (df_csp[col] / df_csp['total_emploi'] * 100).round(3)
df_csp = df_csp[['code_commune','cadres_pct','ouvriers_pct','employes_pct','artisans_pct']]

# --- Diplômes (2022 uniquement) ---
df_dipl = pd.read_sql("""
    SELECT code_commune, nscol15p,
           COALESCE(sup_bac2,0) + COALESCE(sup_bac34,0) + COALESCE(sup_bac5,0) AS bac_plus,
           COALESCE(sans_diplome,0) AS sans_diplome
    FROM silver.diplomes
    WHERE annee = 2022
""", engine)

df_dipl['pct_bac_plus']    = (df_dipl['bac_plus']     / df_dipl['nscol15p'] * 100).round(3)
df_dipl['pct_sans_diplome'] = (df_dipl['sans_diplome'] / df_dipl['nscol15p'] * 100).round(3)
df_dipl = df_dipl[['code_commune','pct_bac_plus','pct_sans_diplome']]

print(f"ref: {len(df_ref)}, naiss: {len(df_nais)}, deces: {len(df_deces)}, rev: {len(df_rev)}, csp: {len(df_csp)}, dipl: {len(df_dipl)}")

ref: 144, naiss: 2431, deces: 2431, rev: 0, csp: 143, dipl: 123


In [8]:
# Base : pivot élections avec annee présidentielle
df_gold = df_pivot.copy()

# Joindre référentiel
df_gold = df_gold.merge(df_ref, on='code_commune', how='left')

# Joindre naissances (même année si dispo, sinon interpoler)
# Les présidentielles sont en 2002, 2007, 2012, 2017, 2022
# Les naissances sont de 2008 à 2024 -> on fait un merge tolérant
# Pour 2002 et 2007, on utilisera l'année la plus proche disponible
def join_closest_year(df_main, df_feat, feat_cols, feat_name):
    """Jointure sur l'année la plus proche disponible dans df_feat."""
    feat_years = sorted(df_feat['annee'].unique())
    rows = []
    for _, row in df_main[['code_commune','annee']].iterrows():
        closest = min(feat_years, key=lambda y: abs(y - row['annee']))
        match = df_feat[
            (df_feat['code_commune'] == row['code_commune']) &
            (df_feat['annee'] == closest)
        ]
        if not match.empty:
            rows.append({**{'code_commune': row['code_commune'], 'annee': row['annee']},
                         **match[feat_cols].iloc[0].to_dict()})
        else:
            rows.append({'code_commune': row['code_commune'], 'annee': row['annee'],
                         **{c: np.nan for c in feat_cols}})
    return df_main.merge(pd.DataFrame(rows), on=['code_commune','annee'], how='left')

# Nais/Deces : merge exact si possible, sinon année proche
df_nais_deces = df_nais.merge(df_deces, on=['code_commune','annee'], how='outer')
df_nais_deces['solde_naturel'] = (
    df_nais_deces['nb_naissances'].fillna(0) - df_nais_deces['nb_deces'].fillna(0)
).astype('Int64')

df_gold = join_closest_year(df_gold, df_nais_deces,
                             ['nb_naissances','nb_deces','solde_naturel'],
                             'nais_deces')

# Revenus : merge le plus proche
if len(df_rev) > 0:
    df_gold = join_closest_year(df_gold, df_rev,
                                 ['mediane_revenu_disp','gini','taux_pauvrete'],
                                 'revenus')
else:
    print("WARN: table revenus vide, colonnes = NaN")
    for c in ['mediane_revenu_disp','gini','taux_pauvrete']:
        df_gold[c] = np.nan

# CSP et Diplômes : statiques 2022, on réplique pour toutes les années
df_gold = df_gold.merge(df_csp,  on='code_commune', how='left', suffixes=('','_csp'))
df_gold = df_gold.merge(df_dipl, on='code_commune', how='left', suffixes=('','_dipl'))

print(f"Gold shape : {df_gold.shape}")
print(f"Colonnes : {df_gold.columns.tolist()}")
df_gold.head(3)

WARN: table revenus vide, colonnes = NaN
Gold shape : (620, 21)
Colonnes : ['code_commune', 'annee', 'pct_centre', 'pct_divers', 'pct_droite', 'pct_gauche', 'bloc_dominant', 'libelle', 'code_dep', 'nb_naissances', 'nb_deces', 'solde_naturel', 'mediane_revenu_disp', 'gini', 'taux_pauvrete', 'cadres_pct', 'ouvriers_pct', 'employes_pct', 'artisans_pct', 'pct_bac_plus', 'pct_sans_diplome']


,code_commune,annee,pct_centre,pct_divers,pct_droite,pct_gauche,bloc_dominant,libelle,code_dep,nb_naissances,...,solde_naturel,mediane_revenu_disp,gini,taux_pauvrete,cadres_pct,ouvriers_pct,employes_pct,artisans_pct,pct_bac_plus,pct_sans_diplome
0,75056,2002,14.534312,0.0,48.408999,37.056689,Droite,PARIS,75,30623.0,...,-29176.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,65.439,11.511
1,75056,2007,20.731619,0.0,41.653697,37.614684,Droite,PARIS,75,30623.0,...,-29176.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,65.439,11.511
2,75056,2012,9.336668,0.0,39.619778,51.043554,Gauche,PARIS,75,29291.0,...,-32143.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,65.439,11.511


In [9]:
# Vérification des taux de remplissage
print("Taux de remplissage (% non-null) :")
fill_rate = (df_gold.notna().sum() / len(df_gold) * 100).round(1)
print(fill_rate.to_string())

Taux de remplissage (% non-null) :
code_commune           100.0
annee                  100.0
pct_centre             100.0
pct_divers             100.0
pct_droite             100.0
pct_gauche             100.0
bloc_dominant          100.0
libelle                100.0
code_dep               100.0
nb_naissances           99.2
nb_deces                99.2
solde_naturel           99.2
mediane_revenu_disp      0.0
gini                     0.0
taux_pauvrete            0.0
cadres_pct              99.2
ouvriers_pct            99.2
employes_pct            99.2
artisans_pct            99.2
pct_bac_plus            99.2
pct_sans_diplome        99.2


## 5. Insertion dans gold.features_communes

In [10]:
# Sélectionner les colonnes attendues par la table Gold
gold_cols = [
    'code_commune', 'annee', 'libelle', 'code_dep',
    'pct_gauche', 'pct_centre', 'pct_droite', 'pct_divers',
    'bloc_dominant',
    'nb_naissances', 'nb_deces', 'solde_naturel',
    'mediane_revenu_disp', 'gini', 'taux_pauvrete',
    'cadres_pct', 'ouvriers_pct', 'employes_pct', 'artisans_pct',
    'pct_bac_plus', 'pct_sans_diplome',
]

df_insert = df_gold[[c for c in gold_cols if c in df_gold.columns]].copy()

# Colonnes manquantes → NaN
for c in gold_cols:
    if c not in df_insert.columns:
        df_insert[c] = np.nan
        print(f"WARN: colonne {c} absente, mise à NaN")

# Types
df_insert['annee'] = df_insert['annee'].astype('Int16')
for col in ['nb_naissances','nb_deces','solde_naturel']:
    if col in df_insert.columns:
        df_insert[col] = df_insert[col].astype('Int64')

print(f"Lignes à insérer : {len(df_insert)}")
df_insert.head()

Lignes à insérer : 620


,code_commune,annee,libelle,code_dep,pct_gauche,pct_centre,pct_droite,pct_divers,bloc_dominant,nb_naissances,...,solde_naturel,mediane_revenu_disp,gini,taux_pauvrete,cadres_pct,ouvriers_pct,employes_pct,artisans_pct,pct_bac_plus,pct_sans_diplome
0,75056,2002,PARIS,75,37.056689,14.534312,48.408999,0.000000,Droite,30623,...,-29176,NaN,NaN,NaN,NaN,NaN,NaN,NaN,65.439,11.511
1,75056,2007,PARIS,75,37.614684,20.731619,41.653697,0.000000,Droite,30623,...,-29176,NaN,NaN,NaN,NaN,NaN,NaN,NaN,65.439,11.511
2,75056,2012,PARIS,75,51.043554,9.336668,39.619778,0.000000,Gauche,29291,...,-32143,NaN,NaN,NaN,NaN,NaN,NaN,NaN,65.439,11.511
3,75056,2017,PARIS,75,30.634085,35.343720,33.885463,0.136732,Centre,27419,...,-36337,NaN,NaN,NaN,NaN,NaN,NaN,NaN,65.439,11.511
4,75056,2022,PARIS,75,42.313248,36.486860,21.199892,0.000000,Gauche,23785,...,-45402,NaN,NaN,NaN,NaN,NaN,NaN,NaN,65.439,11.511


In [11]:
with engine.begin() as conn:
    conn.execute(text("TRUNCATE gold.features_communes"))

df_insert.to_sql(
    'features_communes',
    engine,
    schema='gold',
    if_exists='append',
    index=False,
    method='multi',
    chunksize=1000
)

with engine.connect() as conn:
    n = conn.execute(text("SELECT COUNT(*) FROM gold.features_communes")).scalar()
    dist = conn.execute(text("""
        SELECT annee, bloc_dominant, COUNT(*) AS n
        FROM gold.features_communes
        GROUP BY annee, bloc_dominant
        ORDER BY annee, bloc_dominant
    """)).fetchall()

print(f"gold.features_communes : {n} lignes")
print("\nDistribution blocs dominants par année :")
for row in dist:
    print(f"  {row[0]} | {row[1]:15s} | {row[2]} communes")

gold.features_communes : 620 lignes

Distribution blocs dominants par année :
  2002 | Droite          | 103 communes
  2002 | Gauche          | 21 communes
  2007 | Droite          | 68 communes
  2007 | Gauche          | 56 communes
  2012 | Droite          | 45 communes
  2012 | Gauche          | 79 communes
  2017 | Centre          | 7 communes
  2017 | Droite          | 63 communes
  2017 | Gauche          | 54 communes
  2022 | Centre          | 34 communes
  2022 | Droite          | 10 communes
  2022 | Gauche          | 80 communes


## 6. Validation Gold

In [12]:
df_val = pd.read_sql("""
    SELECT annee,
           COUNT(*) AS nb_communes,
           ROUND(AVG(pct_gauche)::numeric, 1)       AS moy_gauche,
           ROUND(AVG(pct_centre)::numeric, 1)       AS moy_centre,
           ROUND(AVG(pct_droite)::numeric, 1)       AS moy_droite,
           ROUND(AVG(cadres_pct)::numeric, 1)       AS moy_cadres_pct,
           ROUND(AVG(pct_bac_plus)::numeric, 1)     AS moy_bac_plus
    FROM gold.features_communes
    GROUP BY annee
    ORDER BY annee
""", engine)

print("=" * 70)
print("VALIDATION GOLD LAYER - gold.features_communes")
print("=" * 70)
print(df_val.to_string(index=False))
print("\nFeature engineering Gold terminé !")

VALIDATION GOLD LAYER - gold.features_communes
 annee  nb_communes  moy_gauche  moy_centre  moy_droite  moy_cadres_pct  moy_bac_plus
  2002          124        35.9        13.7        50.4            33.1          44.3
  2007          124        37.7        19.3        43.0            33.1          44.3
  2012          124        50.4         8.6        41.0            33.1          44.3
  2017          124        34.0        28.3        37.5            33.1          44.3
  2022          124        44.1        30.0        25.9            33.1          44.3

Feature engineering Gold terminé !
